# 3D Perception Tutorial

We will explore the (1)3D object classification and the (2)instance-level semantic segmentation of indoor scenes. To get to theses goals, this tutorial will guide you through the basic concepts and practical applications of WarpConvNet for point cloud procesing, sparse voxel convolutions, and 3D attention mechanisms.

## Table of Contents
1. Basic Concepts
2. PointCloud Representation
3. Voxel Representation
4. Exercise 1: Voxel Size and Point Cloud Visualization
5. Exercise 2: SparseConv Implementation
6. Exercise 3: 3D Point Cloud Semantic Segmentation with SparseConvNet
7. Exercise 4: 3D Point Cloud Semantic Segmentation with PointTransformerV3


In [38]:
import subprocess
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Optional, Tuple, List
import warnings
warnings.filterwarnings('ignore')

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Python version: 3.10.20 (main, Jun 11 2026, 15:17:37) [GCC 14.3.0]
PyTorch version: 2.5.1
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce RTX 3090


## PointCloud Representation

In [ ]:
import torch

from tools import (
    generate_sample_point_cloud,
    visualize_multiple_point_clouds,
    visualize_point_cloud,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [40]:
shapes = ['sphere', 'cube', 'torus', 'cylinder']
point_clouds = {}

for shape in shapes:
    points, features = generate_sample_point_cloud(2000, shape)
    point_clouds[shape] = points
    
# Visualize individual shape
points_sphere, features_sphere = generate_sample_point_cloud(5000, 'sphere', noise_level=0.02)
print(f"Points shape: {points_sphere.shape}")
print(f"Features shape: {features_sphere.shape}")

# Interactive visualization with color by height
fig = visualize_point_cloud(
    points_sphere, 
    colors=points_sphere[:, 2],  # Color by z-coordinate
    title="Interactive Sphere Point Cloud (5000 points)",
    point_size=2,
    colorscale='Viridis'
)

# Visualize all shapes together for comparison
visualize_multiple_point_clouds(point_clouds, title="Comparison of Different Point Cloud Shapes")


Points shape: (5000, 3)
Features shape: (5000, 3)


In [41]:
from warpconvnet.geometry.types.points import Points

points_tensor = torch.from_numpy(points).to(device)
features_tensor = torch.from_numpy(features).to(device)

# Create batch indices (single batch for now)
batch_indices = torch.zeros(len(points), dtype=torch.long).to(device)

# Create PointCloud object
point_cloud = Points(
    [points_tensor],
    [features_tensor],
)
print(point_cloud)


Points(feature_shape=torch.Size([2000, 3]), coords_shape=torch.Size([2000, 3]))


## Voxel Representation

## Conversion between PointCloud and Voxel

In [ ]:
from tools import visualize_voxels


### Point Clouds -> Voxels

In [43]:
# Create sparse voxel representation with different resolutions
voxel_sizes = [0.05, 0.1, 0.2]
voxel_representations = {}

for voxel_size in voxel_sizes:
    sparse_voxels = point_cloud.to_voxels(voxel_size)
    voxel_representations[f"Size {voxel_size}"] = sparse_voxels
    visualize_voxels(sparse_voxels.coordinates, sparse_voxels.voxel_size)


### Quiz: Voxels -> Point Clouds
- Convert one sparse voxel representation back into a point cloud.
- Check that recovered point coordinates correspond to `voxel_coordinates * voxel_size`.
- reference: https://github.com/NVlabs/WarpConvNet/blob/eda68fa3e3759dddadfc53d76038fd9246bbf885/warpconvnet/geometry/types/voxels.py


In [44]:
# Quiz: convert sparse voxels back to point centers.
# Use the voxel representation with voxel_size = 0.1 created above.

voxel_to_recover = voxel_representations["Size 0.1"]

# Voxels store discrete voxel indices. `to_point` maps each occupied voxel to a
# point coordinate by multiplying those integer coordinates by the voxel size.
recovered_point_cloud = voxel_to_recover.to_point()

manual_coordinates = voxel_to_recover.coordinate_tensor.to(torch.float32) * voxel_to_recover.voxel_size
max_coordinate_error = (recovered_point_cloud.coordinate_tensor - manual_coordinates).abs().max()

print(recovered_point_cloud)
print(f"Voxel coordinates: {voxel_to_recover.coordinate_tensor.shape}")
print(f"Recovered point coordinates: {recovered_point_cloud.coordinate_tensor.shape}")
print(f"Recovered point features: {recovered_point_cloud.feature_tensor.shape}")
print(f"Max coordinate error: {max_coordinate_error.item():.6f}")

assert torch.allclose(recovered_point_cloud.coordinate_tensor, manual_coordinates)

visualize_point_cloud(
    recovered_point_cloud.coordinate_tensor,
    colors=recovered_point_cloud.feature_tensor[:, 0],
    title="Voxel Centers Recovered as Points",
    point_size=4,
)


Points(feature_shape=torch.Size([1431, 3]), coords_shape=torch.Size([1431, 3]))
Voxel coordinates: torch.Size([1431, 3])
Recovered point coordinates: torch.Size([1431, 3])
Recovered point features: torch.Size([1431, 3])
Max coordinate error: 0.000000


## Exercise 1: Scene Selection and Voxel Size Visualization
Choose three ScanNet scenes, save their full-scene PLY files, then choose one scene and three voxel sizes to compare voxelization results.

### Load a Real ScanNet Point Cloud
The representation examples above used synthetic shapes. Now load one real ScanNet scene while keeping the full point cloud available for later voxel-size experiments.


In [ ]:
from pathlib import Path
import os

import torch
from warpconvnet.geometry.types.points import Points

# Locate the ScanNet data directory. Set SCANNET_DATA_DIR to use a different path.
scannet_candidates = []
if os.environ.get("SCANNET_DATA_DIR"):
    scannet_candidates.append(Path(os.environ["SCANNET_DATA_DIR"]))
scannet_candidates.extend([
    Path("/data/scannet_3d"),
    Path("../../data/scannet_3d"),
    Path("data/scannet_3d"),
])

scannet_root = None
for candidate in scannet_candidates:
    if (candidate / "scannetv2_train.txt").exists():
        scannet_root = candidate
        break

if scannet_root is None:
    raise FileNotFoundError(
        "ScanNet data not found. Expected scannetv2_train.txt under /data/scannet_3d, "
        "../../data/scannet_3d, data/scannet_3d, or SCANNET_DATA_DIR."
    )


def scannet_color_features_to_unit_rgb(colors):
    colors = torch.as_tensor(colors, dtype=torch.float32)
    if colors.numel() == 0:
        return colors
    if colors.min() < 0.0:
        colors = (colors + 1.0) * 0.5
    elif colors.max() > 1.0:
        colors = colors / 255.0
    return colors.clamp(0.0, 1.0)


def take_evenly_spaced_points(coords, colors, labels, max_points=None):
    if max_points is None or coords.shape[0] <= max_points:
        return coords, colors, labels
    sample_idx = torch.linspace(0, coords.shape[0] - 1, max_points).long()
    return coords[sample_idx], colors[sample_idx], labels[sample_idx]


device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

scene_name = (scannet_root / "scannetv2_train.txt").read_text().splitlines()[0]
scene_path = scannet_root / "train" / f"{scene_name}_vh_clean_2.pth"
coords_raw, colors_raw, labels_raw = torch.load(scene_path, weights_only=False)

# Keep the full scene in CPU memory first. Downsample only after this point when
# the later demo cells need a smaller active point cloud.
coords_full = torch.as_tensor(coords_raw, dtype=torch.float32)
colors_full = scannet_color_features_to_unit_rgb(colors_raw)
labels_full = torch.as_tensor(labels_raw, dtype=torch.long)

max_scannet_points_for_demo = None  # Set an integer, e.g. 12000, to downsample after full loading.
coords_demo, colors_demo, labels_demo = take_evenly_spaced_points(
    coords_full,
    colors_full,
    labels_full,
    max_points=max_scannet_points_for_demo,
)

coords = coords_demo.to(device)
colors = colors_demo.to(device)
labels = labels_demo.to(device)

scannet_point_cloud = Points([coords], [colors])
point_cloud = scannet_point_cloud
sparse_voxels = point_cloud.to_voxels(0.1)

print(f"Loaded ScanNet scene: {scene_name}")
print(f"Full scene points: {coords_full.shape[0]:,}")
print(f"Active demo points: {point_cloud.coordinate_tensor.shape[0]:,}")
print(f"Sparse voxels at voxel_size=0.1: {sparse_voxels.coordinate_tensor.shape[0]:,}")
print(f"Semantic labels: {labels.shape}")


In [ ]:
from pathlib import Path

from tools import save_scannet_scene_as_ply

# Choose three ScanNet scenes and save each full scene with RGB and semantic colors.
scene_names_to_export = [
    "scene0439_01",
    "scene0065_01",
    "scene0040_00",
]
split_to_export = None  # None searches train/val/test automatically. Or set "train", "val", or "test".
max_export_points = None  # None saves each full scene.
ply_output_dir = Path("ply_exports")
full_scene_color_modes = ["rgb", "semantic"]

full_scene_ply_paths = []
for scene_name_to_export in scene_names_to_export:
    for color_mode in full_scene_color_modes:
        ply_path = save_scannet_scene_as_ply(
            scene_name_to_export,
            split=split_to_export,
            max_points=max_export_points,
            output_dir=ply_output_dir,
            color_mode=color_mode,
            voxel_size=None,
        )
        full_scene_ply_paths.append(ply_path)

full_scene_ply_paths


### Choose Voxel Sizes for One Scene
Select one of the scenes above, then choose three voxel sizes. The same scene will be saved three times with different voxel resolutions.


In [ ]:
# Choose one exported scene and three voxel sizes for visual comparison.
voxelized_scene_name_to_export = scene_names_to_export[0]
voxelized_split_to_export = split_to_export
voxelized_max_export_points = max_export_points
voxelized_ply_output_dir = ply_output_dir
voxelized_export_sizes = [0.05, 0.10, 0.20]
voxelized_export_color_modes = ["rgb", "semantic"]


In [ ]:
from tools import load_scannet_scene_arrays, sample_scene_arrays, voxelize_scene_arrays

# Count how many occupied voxels remain for the selected scene at each voxel size.
loaded_split, selected_coords, selected_colors, selected_labels = load_scannet_scene_arrays(
    voxelized_scene_name_to_export,
    split=voxelized_split_to_export,
    root=scannet_root,
)
selected_coords, selected_colors, selected_labels = sample_scene_arrays(
    selected_coords,
    selected_colors,
    selected_labels,
    max_points=voxelized_max_export_points,
)

voxel_size_summary = []
for voxel_size in voxelized_export_sizes:
    voxel_coords, _, _ = voxelize_scene_arrays(
        selected_coords,
        selected_colors,
        selected_labels,
        voxel_size=voxel_size,
    )
    occupied_voxels = voxel_coords.shape[0]
    voxel_size_summary.append({
        "voxel_size": voxel_size,
        "occupied_voxels": occupied_voxels,
        "compression_ratio": selected_coords.shape[0] / occupied_voxels,
    })

print(f"Selected scene: {voxelized_scene_name_to_export} ({loaded_split})")
print(f"Full scene points: {selected_coords.shape[0]:,}")
print("voxel_size | occupied_voxels | points_per_voxel")
for row in voxel_size_summary:
    print(
        f"{row['voxel_size']:>10.2f} | "
        f"{row['occupied_voxels']:>15,} | "
        f"{row['compression_ratio']:>16.2f}"
    )


In [ ]:
# Save the selected scene at the selected voxelization sizes.
voxelized_ply_paths = []
for voxel_size in voxelized_export_sizes:
    for color_mode in voxelized_export_color_modes:
        ply_path = save_scannet_scene_as_ply(
            voxelized_scene_name_to_export,
            split=voxelized_split_to_export,
            max_points=voxelized_max_export_points,
            output_dir=voxelized_ply_output_dir,
            color_mode=color_mode,
            voxel_size=voxel_size,
        )
        voxelized_ply_paths.append(ply_path)

voxelized_ply_paths


### ScanNet Semantic Color Legend
Use this legend when inspecting semantic-label-colored PLY files.


In [ ]:
from tools import display_scannet_semantic_legend

display_scannet_semantic_legend()


## Exercise 2: SparseConv Implementation
Use WarpConvNet as the reference, implement one submanifold sparse convolution in plain PyTorch-style tensor operations, and compare the output features numerically.


In [ ]:
# 1. WarpConvNet SparseConv3d baseline.
# Use a small deterministic sparse voxel tensor so the custom implementation can be checked exactly.
import torch

from warpconvnet.geometry.types.voxels import Voxels
from warpconvnet.nn.modules.sparse_conv import SparseConv3d


device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

reference_coords = torch.tensor([
    [0, 0, 0],
    [1, 0, 0],
    [-1, 0, 0],
    [0, 1, 0],
    [0, -1, 0],
    [0, 0, 1],
    [0, 0, -1],
    [1, 1, 0],
    [1, 0, 1],
], dtype=torch.int32, device=device)

num_reference_voxels = reference_coords.shape[0]
feature_values = torch.arange(num_reference_voxels * 3, dtype=torch.float32, device=device)
feature_values = feature_values.reshape(num_reference_voxels, 3)
reference_features = (feature_values % 5) / 100.0
reference_voxels = Voxels([reference_coords], [reference_features])

warp_sparse_conv = SparseConv3d(
    3,
    4,
    kernel_size=3,
    stride=1,
    bias=True,
    fwd_algo="explicit_gemm",
    compute_dtype=torch.float32,
).to(device)

with torch.no_grad():
    weight_values = torch.linspace(
        -0.02,
        0.02,
        warp_sparse_conv.weight.numel(),
        device=device,
    )
    weight_values = weight_values.reshape_as(warp_sparse_conv.weight)
    warp_sparse_conv.weight.copy_(weight_values)

    bias_values = torch.linspace(
        -0.01,
        0.01,
        warp_sparse_conv.bias.numel(),
        device=device,
    )
    warp_sparse_conv.bias.copy_(bias_values)

warp_output = warp_sparse_conv(reference_voxels)

print(f"Input coordinates: {reference_voxels.coordinate_tensor.shape}")
print(f"Input features: {reference_voxels.feature_tensor.shape}")
print(f"WarpConvNet output features: {warp_output.feature_tensor.shape}")
print(warp_output.feature_tensor[:3].detach().cpu())


In [ ]:
# 2. Custom reference implementation of submanifold 3D sparse convolution.
# This version uses basic Python loops so each lookup is easy to follow.
def custom_submanifold_sparse_conv3d(input_voxels, weight, bias=None):
    if weight.shape[0] != 27:
        raise ValueError("This reference implementation expects a 3x3x3 kernel.")

    coords = input_voxels.batch_indexed_coordinates.to(torch.int64)
    coords_list = coords.detach().cpu().tolist()
    features = input_voxels.feature_tensor

    num_voxels = features.shape[0]
    out_channels = weight.shape[2]
    out_features = torch.zeros(
        (num_voxels, out_channels),
        dtype=features.dtype,
        device=features.device,
    )

    coord_to_index = {}
    for point_index in range(num_voxels):
        batch_index = coords_list[point_index][0]
        x = coords_list[point_index][1]
        y = coords_list[point_index][2]
        z = coords_list[point_index][3]
        coord_to_index[(batch_index, x, y, z)] = point_index

    kernel_offsets = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                kernel_offsets.append((dx, dy, dz))

    for output_index in range(num_voxels):
        batch_index = coords_list[output_index][0]
        x = coords_list[output_index][1]
        y = coords_list[output_index][2]
        z = coords_list[output_index][3]

        for kernel_index in range(len(kernel_offsets)):
            dx, dy, dz = kernel_offsets[kernel_index]
            neighbor_coord = (batch_index, x + dx, y + dy, z + dz)
            input_index = coord_to_index.get(neighbor_coord)

            if input_index is None:
                continue

            input_feature = features[input_index]
            kernel_weight = weight[kernel_index]
            contribution = input_feature @ kernel_weight
            out_features[output_index] = out_features[output_index] + contribution

    if bias is not None:
        out_features = out_features + bias

    return input_voxels.replace(batched_features=out_features)


In [ ]:
# 3. Compare the custom implementation with WarpConvNet.
custom_output = custom_submanifold_sparse_conv3d(
    reference_voxels,
    warp_sparse_conv.weight,
    warp_sparse_conv.bias,
)

coord_match = torch.equal(
    warp_output.batch_indexed_coordinates.detach().cpu(),
    custom_output.batch_indexed_coordinates.detach().cpu(),
)
max_abs_error = (warp_output.feature_tensor - custom_output.feature_tensor).abs().max()
mean_abs_error = (warp_output.feature_tensor - custom_output.feature_tensor).abs().mean()

print(f"Coordinate match: {coord_match}")
print(f"Max feature error: {max_abs_error.item():.8f}")
print(f"Mean feature error: {mean_abs_error.item():.8f}")
print("WarpConvNet output sample:")
print(warp_output.feature_tensor[:3].detach().cpu())
print("Custom output sample:")
print(custom_output.feature_tensor[:3].detach().cpu())

assert coord_match
assert torch.allclose(warp_output.feature_tensor, custom_output.feature_tensor, atol=1e-6, rtol=1e-6)


## Exercise 3: 3D Point Cloud Semantic Segmentation with SparseConvNet


In [ ]:
from typing import Dict, List, Optional, Tuple, Union
import os
from pathlib import Path
import yaml

try:
    import hydra
    from hydra.core.config_store import ConfigStore
    from omegaconf import DictConfig, OmegaConf
except ImportError:
    print("Hydra and OmegaConf not installed, pip install hydra-core omegaconf")
    exit(1)

import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import warp as wp
from torch import Tensor
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Subset
from torchmetrics.classification import MulticlassConfusionMatrix
from tqdm import tqdm
from warpconvnet.dataset.scannet import ScanNetDataset
from warpconvnet.geometry.base.geometry import Geometry
from warpconvnet.geometry.types.points import Points
from warpconvnet.nn.modules.sparse_pool import PointToSparseWrapper

from mink_unet import MinkUNetBase


In [ ]:
def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def resolve_scannet_data_dir(preferred_path: Optional[Union[str, Path]] = None) -> str:
    candidates = []
    env_path = os.environ.get("SCANNET_DATA_DIR")
    if env_path:
        candidates.append(env_path)
    if preferred_path:
        candidates.append(preferred_path)
    candidates.extend([
        "/data/scannet_3d",
        "../../data/scannet_3d",
        "data/scannet_3d",
    ])

    checked = []
    for candidate in candidates:
        path = Path(candidate).expanduser()
        if not path.is_absolute():
            path = (Path.cwd() / path).resolve()
        if path in checked:
            continue
        checked.append(path)
        if path.exists():
            return str(path)

    checked_text = "\n".join(f"- {path}" for path in checked)
    raise FileNotFoundError(
        "ScanNet data directory was not found. Set SCANNET_DATA_DIR or update cfg.paths.data_dir.\n"
        f"Checked:\n{checked_text}"
    )

def collate_fn(batch: List[Dict[str, Tensor]]):
    """
    Return dict of list of tensors
    """
    keys = batch[0].keys()
    return {key: [torch.tensor(item[key]) for item in batch] for key in keys}


class DataToTensor:
    def __init__(
        self,
        device: str = "cuda",
    ):
        self.device = device

    def __call__(self, batch_dict: Dict[str, Tensor]) -> Tuple[Geometry, Dict[str, Tensor]]:
        # cat all features into a single tensor
        cat_batch_dict = {k: torch.cat(v, dim=0).to(self.device) for k, v in batch_dict.items()}
        return (
            Points.from_list_of_coordinates(
                batch_dict["coords"],
                features=batch_dict["colors"],
            ).to(self.device),
            cat_batch_dict,
        )


def confusion_matrix_to_metrics(conf_matrix: Tensor) -> Dict[str, float]:
    """
    Return accuracy, miou, class_iou, class_accuracy

    Rows are ground truth, columns are predictions.
    """
    conf_matrix = conf_matrix.cpu()
    accuracy = (conf_matrix.diag().sum() / conf_matrix.sum()).item() * 100
    class_accuracy = (conf_matrix.diag() / conf_matrix.sum(dim=1)) * 100
    class_iou = conf_matrix.diag() / (
        conf_matrix.sum(dim=1) + conf_matrix.sum(dim=0) - conf_matrix.diag()
    )
    miou = class_iou.mean().item() * 100
    return {
        "accuracy": accuracy,
        "miou": miou,
        "class_iou": class_iou,
        "class_accuracy": class_accuracy,
    }

from tools import save_latest_visual_data_as_plys

def checkpoint_path_from_cfg(cfg, filename: str) -> Path:
    output_dir = Path(cfg.paths.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / filename


def save_model_checkpoint(model: nn.Module, cfg, filename: str) -> Path:
    checkpoint_path = checkpoint_path_from_cfg(cfg, filename)
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "config": OmegaConf.to_container(cfg, resolve=True),
    }
    torch.save(checkpoint, checkpoint_path)
    print(f"Saved checkpoint: {checkpoint_path}")
    return checkpoint_path


def load_model_checkpoint(model: nn.Module, checkpoint_path: Union[str, Path], device: torch.device):
    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict)
    print(f"Loaded checkpoint: {checkpoint_path}")
    return checkpoint



In [ ]:
@torch.amp.autocast(device_type="cuda", enabled=True)
def train(
    model: nn.Module,
    train_loader: DataLoader,
    optimizer: optim.Optimizer,
    epoch: int,
    cfg: DictConfig,
):
    model.train()
    bar = tqdm(train_loader)
    dict_to_data = DataToTensor(device=cfg.device)
    for batch_dict in bar:
        start_time = time.time()
        optimizer.zero_grad()
        st, batch_dict = dict_to_data(batch_dict)
        output = model(st.to(cfg.device))
        loss = F.cross_entropy(
            output.features,
            batch_dict["labels"].long().to(cfg.device),
            reduction="mean",
            ignore_index=cfg.data.ignore_index,
        )
        loss.backward()
        optimizer.step()
        bar.set_description(f"Train Epoch: {epoch} Loss: {loss.item(): .3f}")


@torch.amp.autocast(device_type="cuda", enabled=True)
@torch.inference_mode()
def test(
    model: nn.Module,
    test_loader: DataLoader,
    cfg: DictConfig,
    num_test_batches: Optional[int] = None,
    save_visuals: bool = False,
    return_visuals: bool = False,
):
    model.eval()
    torch.cuda.empty_cache()
    confusion_matrix = MulticlassConfusionMatrix(
        num_classes=cfg.data.num_classes, ignore_index=cfg.data.ignore_index
    ).to(cfg.device)
    test_loss = 0
    num_batches = 0

    visual_data = []
    dict_to_data = DataToTensor(device=cfg.device)
    for batch_dict in tqdm(test_loader):
        original_batch_dict = batch_dict
        st, batch_dict = dict_to_data(batch_dict)
        output = model(st.to(cfg.device))
        labels = batch_dict["labels"].long().to(cfg.device)
        test_loss += F.cross_entropy(
            output.features,
            labels,
            reduction="mean",
            ignore_index=cfg.data.ignore_index,
        ).item()
        pred = output.features.argmax(dim=1)
        confusion_matrix.update(pred, labels)

        if save_visuals or return_visuals:
            num_items_in_batch = len(st.offsets) - 1
            for i in range(num_items_in_batch):
                start_idx = st.offsets[i]
                end_idx = st.offsets[i+1]

                visual_data.append({
                    "coords": original_batch_dict["coords"][i],
                    "colors": original_batch_dict["colors"][i],
                    "preds": pred[start_idx:end_idx].cpu(),
                    "labels": labels[start_idx:end_idx].cpu(),
                })
        
        num_batches += 1
        if num_test_batches is not None and num_batches >= num_test_batches:
            break

    metrics = confusion_matrix_to_metrics(confusion_matrix.compute())
    
    print(
        f"\nTest set: Average loss: {test_loss / num_batches: .4f}, Accuracy: {metrics['accuracy']: .2f}%, mIoU: {metrics['miou']: .2f}%\n"
    )
    if return_visuals:
        return metrics, visual_data
    return metrics


In [ ]:
# Embedded YAML configuration
CONFIG_YAML_BASE = """
# Path configuration
paths:
  data_dir: /data/scannet_3d
  output_dir: ./results/scannet_3d/base_sparseconv
  ckpt_path: null

# Training configuration.
train:
  batch_size: 32
  lr: 0.005
  epochs: 2
  checkpoint_interval: 5
  step_size: 20
  gamma: 0.7
  num_workers: 8

# Testing configuration
test:
  batch_size: 12
  num_workers: 4

# Dataset configuration
data:
  num_classes: 20
  voxel_size: 0.02
  ignore_index: 255

# Model configuration
model:
  _target_: mink_unet.MinkUNet18
  in_channels: 3
  out_channels: 20
  in_type: "voxel"

# General configuration
device: "cuda"
use_wandb: false
seed: 42
"""


In [ ]:
# Load configs
cell_start_time = time.time()
cfg_dict = yaml.safe_load(CONFIG_YAML_BASE)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = resolve_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

# Define dataloaders
train_dataset = ScanNetDataset(cfg.paths.data_dir, split="train",)
train_dataset = Subset(train_dataset, range(100))
test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = Subset(test_dataset, range(8))
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train.batch_size,
    num_workers=cfg.train.num_workers,
    shuffle=True,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

# Model initialization
model = MinkUNetBase(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

optimizer = optim.AdamW(model.parameters(), lr=cfg.train.lr)
scheduler = StepLR(optimizer, step_size=cfg.train.step_size, gamma=cfg.train.gamma)
checkpoint_interval = int(getattr(cfg.train, "checkpoint_interval", 5))

# Test before training
metrics = test(model, test_loader, cfg, num_test_batches=5)

for epoch in range(1, cfg.train.epochs + 1):
    train(
        model,
        train_loader,
        optimizer,
        epoch,
        cfg,
    )
    metrics = test(model, test_loader, cfg)
    if checkpoint_interval > 0 and epoch % checkpoint_interval == 0 and epoch < cfg.train.epochs:
        save_model_checkpoint(model, cfg, f"base_sparseconv_epoch_{epoch:03d}.pth")
    scheduler.step()

print(f"Final mIoU: {metrics['miou']: .2f}%")

checkpoint_path = save_model_checkpoint(model, cfg, "base_sparseconv_final.pth")
print(f"Training cell runtime: {(time.time() - cell_start_time) / 60:.1f} min")

del model
torch.cuda.empty_cache()


In [ ]:
# Load the saved Base SparseConvNet checkpoint and run inference.
cfg_dict = yaml.safe_load(CONFIG_YAML_BASE)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = resolve_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = Subset(test_dataset, range(8))
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

model = MinkUNetBase(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

checkpoint_path = checkpoint_path_from_cfg(cfg, "base_sparseconv_final.pth")
load_model_checkpoint(model, checkpoint_path, device)
metrics, latest_visual_data = test(model, test_loader, cfg, return_visuals=True)
print(f"Loaded-checkpoint mIoU: {metrics['miou']: .2f}%")

print(f"Collected visual data for {len(latest_visual_data)} point clouds")

del model
torch.cuda.empty_cache()


In [ ]:
# Run this after a checkpoint inference cell has collected latest_visual_data.
from pathlib import Path

# Change this directory to custom_sparseconv or point_transformer_v3 for other model outputs.
visual_output_dir = Path("./results/scannet_3d/base_sparseconv")
max_visualized_samples = 4

semantic_visual_ply_paths = save_latest_visual_data_as_plys(
    latest_visual_data,
    output_dir=visual_output_dir,
    max_items=max_visualized_samples,
)
semantic_visual_ply_paths


In [ ]:
class MinkUNetCustom(MinkUNetBase):
    def __init__(self, in_channels: int, out_channels: int, **kwargs):
        super().__init__(
            in_channels=in_channels,
            out_channels=out_channels,
            init_dim=16,
            planes=(16, 32, 64, 128, 128, 64, 48, 48),
            layers=(1, 1, 1, 1, 1, 1, 1, 1),
            **kwargs,
        )


In [ ]:
# Embedded YAML configuration
CONFIG_YAML_CUSTOM = """
# Path configuration
paths:
  data_dir: /data/scannet_3d
  output_dir: ./results/scannet_3d/custom_sparseconv
  ckpt_path: null

# Training configuration.
train:
  batch_size: 64
  lr: 0.005
  epochs: 2
  checkpoint_interval: 5
  step_size: 20
  gamma: 0.7
  num_workers: 8

# Testing configuration
test:
  batch_size: 12
  num_workers: 4

# Dataset configuration
data:
  num_classes: 20
  voxel_size: 0.02
  ignore_index: 255

# Model configuration
model:
  _target_: mink_unet.MinkUNet18
  in_channels: 3
  out_channels: 20
  in_type: "voxel"

# General configuration
device: "cuda"
use_wandb: false
seed: 42
"""


In [ ]:
# Load configs
cell_start_time = time.time()
cfg_dict = yaml.safe_load(CONFIG_YAML_CUSTOM)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = resolve_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

# Define dataloaders
train_dataset = ScanNetDataset(cfg.paths.data_dir, split="train",)
train_dataset = Subset(train_dataset, range(100))
test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = Subset(test_dataset, range(8))
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train.batch_size,
    num_workers=cfg.train.num_workers,
    shuffle=True,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

# Model initialization
model = MinkUNetCustom(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

optimizer = optim.AdamW(model.parameters(), lr=cfg.train.lr)
scheduler = StepLR(optimizer, step_size=cfg.train.step_size, gamma=cfg.train.gamma)
checkpoint_interval = int(getattr(cfg.train, "checkpoint_interval", 5))

# Test before training
metrics = test(model, test_loader, cfg, num_test_batches=5)

for epoch in range(1, cfg.train.epochs + 1):
    train(
        model,
        train_loader,
        optimizer,
        epoch,
        cfg,
    )
    metrics = test(model, test_loader, cfg)
    if checkpoint_interval > 0 and epoch % checkpoint_interval == 0 and epoch < cfg.train.epochs:
        save_model_checkpoint(model, cfg, f"custom_sparseconv_epoch_{epoch:03d}.pth")
    scheduler.step()

print(f"Final mIoU: {metrics['miou']: .2f}%")

checkpoint_path = save_model_checkpoint(model, cfg, "custom_sparseconv_final.pth")
print(f"Training cell runtime: {(time.time() - cell_start_time) / 60:.1f} min")

del model
torch.cuda.empty_cache()


In [ ]:
# Load the saved Custom SparseConvNet checkpoint and run inference.
cfg_dict = yaml.safe_load(CONFIG_YAML_CUSTOM)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = resolve_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = Subset(test_dataset, range(8))
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

model = MinkUNetCustom(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

checkpoint_path = checkpoint_path_from_cfg(cfg, "custom_sparseconv_final.pth")
load_model_checkpoint(model, checkpoint_path, device)
metrics, latest_visual_data = test(model, test_loader, cfg, return_visuals=True)
print(f"Loaded-checkpoint mIoU: {metrics['miou']: .2f}%")

print(f"Collected visual data for {len(latest_visual_data)} point clouds")

del model
torch.cuda.empty_cache()


## Exercise 4: 3D Point Cloud Semantic Segmentation with PointTransformerV3


In [ ]:
# Embedded YAML configuration
CONFIG_YAML_PTV3 = """
# Path configuration
paths:
  data_dir: /data/scannet_3d
  output_dir: ./results/scannet_3d/point_transformer_v3
  ckpt_path: null

# Training configuration.
train:
  batch_size: 4
  lr: 0.001
  epochs: 2
  checkpoint_interval: 5
  step_size: 20
  gamma: 0.7
  num_workers: 8

# Testing configuration
test:
  batch_size: 8
  num_workers: 4

# Dataset configuration
data:
  num_classes: 20
  voxel_size: 0.02
  ignore_index: 255

# Model configuration
model:
  _target_: mink_unet.MinkUNet18
  in_channels: 3
  out_channels: 20
  in_type: "voxel"

# General configuration
device: "cuda"
use_wandb: false
seed: 42
"""


In [ ]:
from point_transformer_v3 import PointTransformerV3

# Load configs
cell_start_time = time.time()
cfg_dict = yaml.safe_load(CONFIG_YAML_PTV3)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = resolve_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

# Define dataloaders
train_dataset = ScanNetDataset(cfg.paths.data_dir, split="train",)
train_dataset = Subset(train_dataset, range(100))
test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = Subset(test_dataset, range(8))
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train.batch_size,
    num_workers=cfg.train.num_workers,
    shuffle=True,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

# Model initialization
model = PointTransformerV3(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

optimizer = optim.AdamW(model.parameters(), lr=cfg.train.lr)
scheduler = StepLR(optimizer, step_size=cfg.train.step_size, gamma=cfg.train.gamma)
checkpoint_interval = int(getattr(cfg.train, "checkpoint_interval", 5))

# Test before training
metrics = test(model, test_loader, cfg, num_test_batches=5)

for epoch in range(1, cfg.train.epochs + 1):
    train(
        model,
        train_loader,
        optimizer,
        epoch,
        cfg,
    )
    metrics = test(model, test_loader, cfg)
    if checkpoint_interval > 0 and epoch % checkpoint_interval == 0 and epoch < cfg.train.epochs:
        save_model_checkpoint(model, cfg, f"ptv3_epoch_{epoch:03d}.pth")
    scheduler.step()

print(f"Final mIoU: {metrics['miou']: .2f}%")

checkpoint_path = save_model_checkpoint(model, cfg, "ptv3_final.pth")
print(f"Training cell runtime: {(time.time() - cell_start_time) / 60:.1f} min")

del model
torch.cuda.empty_cache()


In [ ]:
# Load the saved PointTransformerV3 checkpoint and run inference.
from point_transformer_v3 import PointTransformerV3

cfg_dict = yaml.safe_load(CONFIG_YAML_PTV3)
cfg = OmegaConf.create(cfg_dict)
cfg.paths.data_dir = resolve_scannet_data_dir(cfg.paths.data_dir)
print(f"Using ScanNet data: {cfg.paths.data_dir}")

set_seed(cfg.seed)
device = torch.device(cfg.device)

test_dataset = ScanNetDataset(cfg.paths.data_dir, split="val",)
test_dataset = Subset(test_dataset, range(8))
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.test.batch_size,
    num_workers=cfg.test.num_workers,
    shuffle=False,
    collate_fn=collate_fn,
)

model = PointTransformerV3(
    in_channels=cfg.model.in_channels,
    out_channels=cfg.model.out_channels,
).to(device)

if hasattr(cfg.model, "in_type") and cfg.model.in_type == "voxel":
    model = PointToSparseWrapper(
        inner_module=model,
        voxel_size=cfg.data.voxel_size,
        concat_unpooled_pc=False,
    )

checkpoint_path = checkpoint_path_from_cfg(cfg, "ptv3_final.pth")
load_model_checkpoint(model, checkpoint_path, device)
metrics, latest_visual_data = test(model, test_loader, cfg, return_visuals=True)
print(f"Loaded-checkpoint mIoU: {metrics['miou']: .2f}%")

print(f"Collected visual data for {len(latest_visual_data)} point clouds")

del model
torch.cuda.empty_cache()
